In [ ]:
import xarray as xr

# Paths (
#  input)
era5_instant_path = "era-5-2020/data_stream-oper_stepType-instant.nc"
era5_accum_path   = "era-5-2020/data_stream-oper_stepType-accum.nc"
rain_01_path      = "rainfall_2020_01degree_combined.nc"

# Open datasets (lazy loading, no memory blow-up)
ds_era5_inst = xr.open_dataset(era5_instant_path)
ds_era5_acc  = xr.open_dataset(era5_accum_path)
ds_rain_01   = xr.open_dataset(rain_01_path)

print("✅ Files loaded successfully")

✅ Files loaded successfully


In [2]:
print("\n" + "="*70)
print("ERA5 INSTANTANEOUS FILE STRUCTURE")
print("="*70)

print(ds_era5_inst)

print("\n" + "="*70)
print("ERA5 ACCUMULATED FILE STRUCTURE")
print("="*70)

print(ds_era5_acc)

print("\n" + "="*70)
print("0.1° RAINFALL FILE STRUCTURE (TARGET)")
print("="*70)

print(ds_rain_01)


ERA5 INSTANTANEOUS FILE STRUCTURE
<xarray.Dataset> Size: 914MB
Dimensions:     (valid_time: 2928, latitude: 129, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 23kB 2020-01-01 ... 2020-12-31T21...
  * latitude    (latitude) float64 1kB 38.0 37.75 37.5 37.25 ... 6.5 6.25 6.0
  * longitude   (longitude) float64 968B 68.0 68.25 68.5 ... 97.5 97.75 98.0
    number      int64 8B ...
    expver      (valid_time) <U4 47kB ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 183MB ...
    v10         (valid_time, latitude, longitude) float32 183MB ...
    d2m         (valid_time, latitude, longitude) float32 183MB ...
    t2m         (valid_time, latitude, longitude) float32 183MB ...
    sp          (valid_time, latitude, longitude) float32 183MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
 

In [3]:
rain_01_india = ds_rain_01["precipitation"].sel(
    lat=slice(38, 6),      # note descending lat
    lon=slice(68, 98)
)
import numpy as np
# Rename ERA5 time dimension
ds_era5_inst = ds_era5_inst.rename({"valid_time": "time"})
ds_era5_acc  = ds_era5_acc.rename({"valid_time": "time"})

# Ensure exact temporal overlap
common_times = np.intersect1d(
    ds_era5_inst.time.values,
    rain_01_india.time.values
)

ds_era5_inst = ds_era5_inst.sel(time=common_times)
ds_era5_acc  = ds_era5_acc.sel(time=common_times)
rain_01_india = rain_01_india.sel(time=common_times)

print("✅ Temporal alignment done:", len(common_times), "steps")

era5_india = ds_era5_inst.sel(
    latitude=slice(38, 6),
    longitude=slice(68, 98)
)

era5_tp_india = ds_era5_acc["tp"].sel(
    latitude=slice(38, 6),
    longitude=slice(68, 98)
)

✅ Temporal alignment done: 2928 steps


In [4]:
import numpy as np

# Coarsen ERA5 instantaneous variables from 0.25° to 0.5°
era5_india_05 = era5_india.coarsen(latitude=2, longitude=2, boundary='trim').mean()

# Coarsen ERA5 precipitation from 0.25° to 0.5°
era5_tp_india_05 = era5_tp_india.coarsen(latitude=2, longitude=2, boundary='trim').mean()

print("Original ERA5 shape:", era5_india.dims)
print("Coarsened to 0.5° shape:", era5_india_05.dims)
print("\nOriginal ERA5 precipitation shape:", era5_tp_india.shape)
print("Coarsened to 0.5° shape:", era5_tp_india_05.shape)

# Verify the new resolution
print("\nNew latitude resolution:", np.diff(era5_india_05.latitude.values[:5]))
print("New longitude resolution:", np.diff(era5_india_05.longitude.values[:5]))

Original ERA5 shape: FrozenMappingWarningOnValuesAccess({'time': 2928, 'latitude': 129, 'longitude': 121})
Coarsened to 0.5° shape: FrozenMappingWarningOnValuesAccess({'time': 2928, 'latitude': 64, 'longitude': 60})

Original ERA5 precipitation shape: (2928, 129, 121)
Coarsened to 0.5° shape: (2928, 64, 60)

New latitude resolution: [-0.5 -0.5 -0.5 -0.5]
New longitude resolution: [0.5 0.5 0.5 0.5]


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from scipy.stats import iqr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from collections import defaultdict

# ============================================================================
# STEP 1: Prepare inputs - Use 0.1° rainfall downsampled to 0.5°
# ============================================================================

# Coarsen ERA5 meteorological variables to 0.5°
era5_india_05 = era5_india.coarsen(latitude=2, longitude=2, boundary='trim').mean()

# NEW: Downsample 0.1° rainfall to 0.5° (instead of using ERA5 TP)
print("Downsampling 0.1° rainfall to 0.5° for input...")

# Calculate downsampling factor: 0.5° / 0.1° = 5x
# We need to go from fine (0.1°) to coarse (0.5°)
rain_01_tensor = torch.FloatTensor(rain_01_india.values).unsqueeze(1)  # (T, 1, H, W)

# Use average pooling to downsample 5x
rain_05_downsampled = F.avg_pool2d(
    rain_01_tensor, 
    kernel_size=5, 
    stride=5
)  # (T, 1, H_new, W_new)

rain_05_downsampled = rain_05_downsampled.squeeze(1).numpy()  # (T, H, W)

print(f"Original 0.1° rainfall shape: {rain_01_india.shape}")
print(f"Downsampled 0.5° rainfall shape: {rain_05_downsampled.shape}")
print(f"ERA5 0.5° shape: {era5_india_05['u10'].shape}")

# Verify they match
assert rain_05_downsampled.shape[1:] == era5_india_05['u10'].shape[1:], \
    "Spatial dimensions must match!"

# Stack features with downsampled rainfall instead of ERA5 TP
X_features = np.stack([
    era5_india_05['u10'].values,      # 10m U wind (m/s)
    era5_india_05['v10'].values,      # 10m V wind (m/s)
    era5_india_05['t2m'].values,      # 2m temperature (K)
    era5_india_05['d2m'].values,      # 2m dewpoint (K)
    era5_india_05['sp'].values,       # surface pressure (Pa)
    rain_05_downsampled               # 0.1° rainfall downsampled to 0.5° (mm/3h)
], axis=-1)  # Shape: (T, H, W, 6)


X_features = np.transpose(X_features, (0, 3, 1, 2))
y_target = rain_01_india.values

# Identify extreme rainfall events for oversampling
extreme_threshold = np.percentile(y_target[y_target > 0], 90)  # 90th percentile
extreme_events = np.any(y_target > extreme_threshold, axis=(1, 2))
print(f"Extreme events (>{extreme_threshold:.2f}mm): {extreme_events.sum()} / {len(y_target)}")


# Store coordinate information for plotting (auto-detect coordinate names)
# Try common coordinate names
coord_names = list(rain_01_india.coords.keys())
lat_name = [c for c in coord_names if c in ['latitude', 'lat', 'y']][0]
lon_name = [c for c in coord_names if c in ['longitude', 'lon', 'x']][0]
lat_coords = rain_01_india.coords[lat_name].values
lon_coords = rain_01_india.coords[lon_name].values
print(f"   Detected coordinates: {lat_name}, {lon_name}")

# ============================================================================
# STEP 2: Normalization
# ============================================================================

X_normalized = np.zeros_like(X_features, dtype=np.float32)

for i in range(5):
    median = np.median(X_features[:, i])
    iqr_val = iqr(X_features[:, i], axis=None)
    X_normalized[:, i] = (X_features[:, i] - median) / (iqr_val + 1e-8)

X_normalized[:, 5] = np.log1p(X_features[:, 5])

print(f"\n✅ Normalization complete")

# ============================================================================
# STEP 3: Dataset
# ============================================================================

class RainfallDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        self.augment = augment
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        
        if self.augment and torch.rand(1) > 0.5:
            x = torch.flip(x, [2])
            y = torch.flip(y, [1])
        
        return x, y

train_size = int(0.8 * len(X_normalized))
X_train, X_val = X_normalized[:train_size], X_normalized[train_size:]
y_train, y_val = y_target[:train_size], y_target[train_size:]

train_dataset = RainfallDataset(X_train, y_train, augment=True)
val_dataset = RainfallDataset(X_val, y_val, augment=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)

# ============================================================================
# STEP 4: Simple U-Net (WORKING BASELINE)
# ============================================================================

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

class UpBlock(nn.Module):
    def __init__(self, in_ch, out_ch, scale_factor=2):
        super().__init__()
        self.up = nn.Upsample(scale_factor=scale_factor, mode='bilinear', align_corners=False)
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        x = self.up(x)
        return self.conv(x)

class RainfallUNet(nn.Module):
    def __init__(self, in_channels=6, base_filters=64):
        super().__init__()
        
        # Encoder
        self.enc1 = DownBlock(in_channels, base_filters)
        self.enc2 = DownBlock(base_filters, base_filters * 2)
        self.enc3 = DownBlock(base_filters * 2, base_filters * 4)
        
        # Bottleneck
        self.bottleneck = DownBlock(base_filters * 4, base_filters * 8)
        
        # Decoder
        self.up1 = UpBlock(base_filters * 8, base_filters * 4, scale_factor=2.5)
        self.up2 = UpBlock(base_filters * 4, base_filters * 2, scale_factor=2)
        self.up3 = UpBlock(base_filters * 2, base_filters, scale_factor=1)
        
        # Three output heads (Bernoulli-Gamma)
        self.occurrence_head = nn.Sequential(
            nn.Conv2d(base_filters, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )
        
        self.shape_head = nn.Sequential(
            nn.Conv2d(base_filters, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )
        
        self.rate_head = nn.Sequential(
            nn.Conv2d(base_filters, 32, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, 1)
        )
        
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        
        # Bottleneck
        b = self.bottleneck(e3)
        
        # Decoder
        d1 = self.up1(b)
        d2 = self.up2(d1)
        d3 = self.up3(d2)
        
        # Resize to exact target size
        d3 = F.interpolate(d3, size=(320, 300), mode='bilinear', align_corners=False)
        
        # Outputs
        occurrence = self.occurrence_head(d3)
        shape = self.shape_head(d3)
        rate = self.rate_head(d3)
        
        return occurrence, shape, rate
    
    def predict_rainfall(self, x):
        with torch.no_grad():
            occurrence, shape, rate = self.forward(x)
            
            p_rain = torch.sigmoid(occurrence)
            shape_pos = F.softplus(shape) + 1.0
            rate_pos = F.softplus(rate) + 1e-6
            intensity = shape_pos / rate_pos
            
            rainfall = p_rain * intensity
            return rainfall.squeeze(1)

# ============================================================================
# STEP 5: Bernoulli-Gamma Loss
# ============================================================================

class BernoulliGammaLoss(nn.Module):
    def __init__(self, occurrence_weight=1.0, intensity_weight=1.0, 
                 extreme_weight=3.0, eps=1e-6, rain_threshold=0.1):
        super().__init__()
        self.occurrence_weight = occurrence_weight
        self.intensity_weight = intensity_weight
        self.extreme_weight = extreme_weight
        self.eps = eps
        self.rain_threshold = rain_threshold
        
    def forward(self, pred_occurrence, pred_shape, pred_rate, target):
        target = target.unsqueeze(1)
        rain_mask = (target > self.rain_threshold).float()
        
        # Bernoulli loss
        bce_loss = F.binary_cross_entropy_with_logits(
            pred_occurrence, rain_mask, reduction='mean'
        )
        
        # Gamma loss
        shape = F.softplus(pred_shape) + 1.0
        rate = F.softplus(pred_rate) + self.eps
        
        if rain_mask.sum() > 0:
            target_rain = target[rain_mask > 0.5]
            shape_rain = shape[rain_mask > 0.5]
            rate_rain = rate[rain_mask > 0.5]
            
            gamma_nll = (
                -shape_rain * torch.log(rate_rain + self.eps) +
                torch.lgamma(shape_rain) -
                (shape_rain - 1) * torch.log(target_rain + self.eps) +
                rate_rain * target_rain
            )
            
            # Weight extreme events
            extreme_threshold = torch.quantile(target_rain, 0.85)
            weights = torch.ones_like(gamma_nll)
            weights[target_rain > extreme_threshold] = self.extreme_weight
            
            gamma_loss = (gamma_nll * weights).mean()
        else:
            gamma_loss = torch.tensor(0.0, device=target.device)
        
        total_loss = (
            self.occurrence_weight * bce_loss +
            self.intensity_weight * gamma_loss
        )
        
        return total_loss, bce_loss, gamma_loss

# ============================================================================
# STEP 6: Metrics including CRPS
# ============================================================================

def compute_crps_simple(ensemble_preds, targets):
    """Simplified CRPS for ensemble predictions"""
    n_ensemble = ensemble_preds.shape[0]
    
    # Term 1: E[|X - Y|]
    term1 = np.abs(ensemble_preds - targets[None, :]).mean(axis=0)
    
    # Term 2: 0.5 * E[|X - X'|]
    diff_sum = 0
    for i in range(n_ensemble):
        for j in range(n_ensemble):
            diff_sum += np.abs(ensemble_preds[i] - ensemble_preds[j])
    term2 = 0.5 * diff_sum / (n_ensemble * n_ensemble)
    
    crps = (term1 - term2).mean()
    return crps

def compute_spatial_crps(ensemble_preds, targets, spatial_shape):
    """Compute CRPS for each spatial location"""
    n_ensemble = ensemble_preds.shape[0]
    n_samples = ensemble_preds.shape[1]
    
    # Reshape to spatial grid
    targets_2d = targets.reshape(spatial_shape)
    crps_map = np.zeros(spatial_shape)
    
    for i in range(spatial_shape[0]):
        for j in range(spatial_shape[1]):
            idx = i * spatial_shape[1] + j
            if idx < len(targets):
                ensemble_at_loc = ensemble_preds[:, idx]
                target_at_loc = targets[idx]
                
                # Term 1: E[|X - Y|]
                term1 = np.abs(ensemble_at_loc - target_at_loc).mean()
                
                # Term 2: 0.5 * E[|X - X'|]
                term2 = 0.5 * np.abs(ensemble_at_loc[:, None] - ensemble_at_loc[None, :]).mean()
                
                crps_map[i, j] = term1 - term2
    
    return crps_map

# ============================================================================
# STEP 7: Training
# ============================================================================

def train_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    total_loss = 0
    total_bce = 0
    total_gamma = 0
    
    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        
        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type='cuda'):
            occurrence, shape, rate = model(X_batch)
            loss, bce_loss, gamma_loss = criterion(occurrence, shape, rate, y_batch)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        total_bce += bce_loss.item()
        total_gamma += gamma_loss.item()
    
    n = len(loader)
    return total_loss/n, total_bce/n, total_gamma/n

def validate_with_metrics(model, loader, device, n_samples=5):
    """Generate ensemble by adding noise to predictions"""
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            
            # Generate ensemble by running model multiple times with dropout
            ensemble = []
            for _ in range(n_samples):
                pred = model.predict_rainfall(X_batch)
                # Add small noise for diversity
                pred = pred + torch.randn_like(pred) * 0.1 * pred
                pred = torch.clamp(pred, min=0)
                ensemble.append(pred.cpu().numpy())
            
            ensemble = np.stack(ensemble, axis=0)  # (n_samples, B, H, W)
            pred_mean = ensemble.mean(axis=0)
            
            all_preds.append(pred_mean)
            all_targets.append(y_batch.numpy())
    
    preds = np.concatenate(all_preds, axis=0).flatten()
    targets = np.concatenate(all_targets, axis=0).flatten()
    
    # Compute metrics
    mae = np.mean(np.abs(preds - targets))
    rmse = np.sqrt(np.mean((preds - targets) ** 2))
    
    ss_res = np.sum((targets - preds) ** 2)
    ss_tot = np.sum((targets - targets.mean()) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    # CRPS (simplified)
    # Reshape for CRPS: (n_samples, total_pixels)
    all_ensemble = []
    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(device)
            ensemble = []
            for _ in range(n_samples):
                pred = model.predict_rainfall(X_batch)
                pred = pred + torch.randn_like(pred) * 0.1 * pred
                pred = torch.clamp(pred, min=0)
                ensemble.append(pred.cpu().numpy().flatten())
            all_ensemble.append(np.stack(ensemble, axis=0))
    
    all_ensemble = np.concatenate(all_ensemble, axis=1)  # (n_samples, total_pixels)
    crps = compute_crps_simple(all_ensemble, targets)
    
    return {
        'mae': mae,
        'rmse': rmse,
        'r2': r2,
        'crps': crps
    }

# ============================================================================
# VISUALIZATION FUNCTIONS
# ============================================================================

def create_rainfall_colormap():
    """Create a beautiful rainfall colormap"""
    colors = [
        '#FFFFFF',  # White (no rain)
        '#E0F3DB',  # Very light green
        '#A8DDB5',  # Light green
        '#7BCCC4',  # Cyan
        '#4EB3D3',  # Light blue
        '#2B8CBE',  # Blue
        '#0868AC',  # Dark blue
        '#084081',  # Navy
        '#810F7C',  # Purple
        '#8B0000',  # Dark red
    ]
    n_bins = 100
    cmap = LinearSegmentedColormap.from_list('rainfall', colors, N=n_bins)
    return cmap

def plot_training_metrics(history, save_path='training_metrics.png'):
    """Plot training metrics including CRPS over epochs"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    epochs = range(len(history['train_loss']))
    
    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', label='Total Loss', linewidth=2)
    axes[0, 0].plot(epochs, history['train_bce'], 'r--', label='BCE Loss', linewidth=1.5)
    axes[0, 0].plot(epochs, history['train_gamma'], 'g--', label='Gamma Loss', linewidth=1.5)
    axes[0, 0].set_xlabel('Epoch', fontsize=12)
    axes[0, 0].set_ylabel('Loss', fontsize=12)
    axes[0, 0].set_title('Training Loss Components', fontsize=14, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # MAE - plot at the epochs where validation was run
    if len(history['val_mae']) > 0:
        # Validation happens every 5 epochs starting from epoch 0
        val_epochs = [i for i in range(len(history['train_loss'])) if i % 5 == 0][:len(history['val_mae'])]
        axes[0, 1].plot(val_epochs, history['val_mae'], 'b-o', linewidth=2, markersize=4)
        axes[0, 1].set_xlabel('Epoch', fontsize=12)
        axes[0, 1].set_ylabel('MAE (mm/3h)', fontsize=12)
        axes[0, 1].set_title('Validation MAE', fontsize=14, fontweight='bold')
        axes[0, 1].grid(True, alpha=0.3)
    
    # R²
    if len(history['val_r2']) > 0:
        val_epochs = [i for i in range(len(history['train_loss'])) if i % 5 == 0][:len(history['val_r2'])]
        axes[1, 0].plot(val_epochs, history['val_r2'], 'g-o', linewidth=2, markersize=4)
        axes[1, 0].set_xlabel('Epoch', fontsize=12)
        axes[1, 0].set_ylabel('R²', fontsize=12)
        axes[1, 0].set_title('Validation R² Score', fontsize=14, fontweight='bold')
        axes[1, 0].grid(True, alpha=0.3)
        axes[1, 0].axhline(y=0, color='r', linestyle='--', alpha=0.5)
    
    # CRPS
    if len(history['val_crps']) > 0:
        val_epochs = [i for i in range(len(history['train_loss'])) if i % 5 == 0][:len(history['val_crps'])]
        axes[1, 1].plot(val_epochs, history['val_crps'], color='purple', marker='o', linewidth=2, markersize=4)
        axes[1, 1].set_xlabel('Epoch', fontsize=12)
        axes[1, 1].set_ylabel('CRPS', fontsize=12)
        axes[1, 1].set_title('Validation CRPS (Lower is Better)', fontsize=14, fontweight='bold')
        axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"📊 Training metrics plot saved to {save_path}")
    plt.close()

def plot_india_rainfall_prediction(model, val_loader, device, lat_coords, lon_coords, 
                                   sample_idx=0, save_path='india_rainfall_map.png'):
    """Plot rainfall prediction on Indian map with CRPS"""
    model.eval()
    
    # Get a sample prediction
    with torch.no_grad():
        for i, (X_batch, y_batch) in enumerate(val_loader):
            if i == sample_idx:
                X_batch = X_batch.to(device)
                
                # Generate ensemble
                n_ensemble = 10
                ensemble = []
                for _ in range(n_ensemble):
                    pred = model.predict_rainfall(X_batch)
                    pred = pred + torch.randn_like(pred) * 0.1 * pred
                    pred = torch.clamp(pred, min=0)
                    ensemble.append(pred.cpu().numpy())
                
                ensemble = np.stack(ensemble, axis=0)  # (n_ensemble, batch, H, W)
                
                # Take first sample from batch
                pred_mean = ensemble.mean(axis=0)[0]  # (H, W)
                target = y_batch[0].numpy()  # (H, W)
                
                # Compute spatial CRPS
                ensemble_sample = ensemble[:, 0, :, :]  # (n_ensemble, H, W)
                ensemble_flat = ensemble_sample.reshape(n_ensemble, -1)
                target_flat = target.flatten()
                spatial_shape = target.shape
                crps_map = compute_spatial_crps(ensemble_flat, target_flat, spatial_shape)
                
                break
    
    # Create figure with Indian map projection
    fig = plt.figure(figsize=(20, 14))
    
    # Define India bounds
    india_extent = [lon_coords.min(), lon_coords.max(), 
                    lat_coords.min(), lat_coords.max()]
    
    # Subplot 1: Ground Truth
    ax1 = plt.subplot(2, 2, 1, projection=ccrs.PlateCarree())
    ax1.set_extent(india_extent, crs=ccrs.PlateCarree())
    ax1.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax1.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor='black')
    ax1.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray', alpha=0.5)
    
    cmap = create_rainfall_colormap()
    vmax = max(target.max(), pred_mean.max())
    
    im1 = ax1.pcolormesh(lon_coords, lat_coords, target, 
                         transform=ccrs.PlateCarree(), 
                         cmap=cmap, vmin=0, vmax=vmax)
    ax1.set_title('Ground Truth Rainfall', fontsize=16, fontweight='bold', pad=20)
    cbar1 = plt.colorbar(im1, ax=ax1, orientation='horizontal', pad=0.05, shrink=0.8)
    cbar1.set_label('Rainfall (mm/3h)', fontsize=12)
    
    # Subplot 2: Prediction
    ax2 = plt.subplot(2, 2, 2, projection=ccrs.PlateCarree())
    ax2.set_extent(india_extent, crs=ccrs.PlateCarree())
    ax2.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax2.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor='black')
    ax2.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray', alpha=0.5)
    
    im2 = ax2.pcolormesh(lon_coords, lat_coords, pred_mean, 
                         transform=ccrs.PlateCarree(), 
                         cmap=cmap, vmin=0, vmax=vmax)
    ax2.set_title('Model Prediction', fontsize=16, fontweight='bold', pad=20)
    cbar2 = plt.colorbar(im2, ax=ax2, orientation='horizontal', pad=0.05, shrink=0.8)
    cbar2.set_label('Rainfall (mm/3h)', fontsize=12)
    
    # Subplot 3: Absolute Error
    ax3 = plt.subplot(2, 2, 3, projection=ccrs.PlateCarree())
    ax3.set_extent(india_extent, crs=ccrs.PlateCarree())
    ax3.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax3.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor='black')
    ax3.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray', alpha=0.5)
    
    error = np.abs(pred_mean - target)
    im3 = ax3.pcolormesh(lon_coords, lat_coords, error, 
                         transform=ccrs.PlateCarree(), 
                         cmap='YlOrRd', vmin=0, vmax=error.max())
    ax3.set_title('Absolute Error', fontsize=16, fontweight='bold', pad=20)
    cbar3 = plt.colorbar(im3, ax=ax3, orientation='horizontal', pad=0.05, shrink=0.8)
    cbar3.set_label('Error (mm/3h)', fontsize=12)
    
    # Subplot 4: Spatial CRPS
    ax4 = plt.subplot(2, 2, 4, projection=ccrs.PlateCarree())
    ax4.set_extent(india_extent, crs=ccrs.PlateCarree())
    ax4.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax4.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor='black')
    ax4.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray', alpha=0.5)
    
    im4 = ax4.pcolormesh(lon_coords, lat_coords, crps_map, 
                         transform=ccrs.PlateCarree(), 
                         cmap='viridis', vmin=0, vmax=np.percentile(crps_map, 95))
    ax4.set_title('Spatial CRPS (Probabilistic Skill)', fontsize=16, fontweight='bold', pad=20)
    cbar4 = plt.colorbar(im4, ax=ax4, orientation='horizontal', pad=0.05, shrink=0.8)
    cbar4.set_label('CRPS (Lower is Better)', fontsize=12)
    
    # Overall title with statistics
    mae = np.mean(error)
    rmse = np.sqrt(np.mean(error**2))
    mean_crps = np.mean(crps_map)
    
    fig.suptitle(f'Rainfall Prediction Over India | MAE: {mae:.3f} mm | RMSE: {rmse:.3f} mm | CRPS: {mean_crps:.4f}',
                 fontsize=18, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"🗺️  India rainfall map saved to {save_path}")
    plt.close()

def plot_multiple_samples_india(model, val_loader, device, lat_coords, lon_coords, 
                                n_samples=4, save_path='india_rainfall_samples.png'):
    """Plot multiple rainfall prediction samples on Indian map"""
    model.eval()
    
    fig = plt.figure(figsize=(24, 6 * n_samples))
    
    cmap = create_rainfall_colormap()
    india_extent = [lon_coords.min(), lon_coords.max(), 
                    lat_coords.min(), lat_coords.max()]
    
    sample_count = 0
    
    with torch.no_grad():
        for batch_idx, (X_batch, y_batch) in enumerate(val_loader):
            if sample_count >= n_samples:
                break
                
            X_batch = X_batch.to(device)
            
            for i in range(X_batch.shape[0]):
                if sample_count >= n_samples:
                    break
                
                # Get prediction
                pred = model.predict_rainfall(X_batch[i:i+1])[0].cpu().numpy()
                target = y_batch[i].numpy()
                
                vmax = max(target.max(), pred.max())
                
                # Ground Truth
                ax1 = plt.subplot(n_samples, 3, sample_count * 3 + 1, 
                                 projection=ccrs.PlateCarree())
                ax1.set_extent(india_extent, crs=ccrs.PlateCarree())
                ax1.add_feature(cfeature.COASTLINE, linewidth=0.6)
                ax1.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor='black')
                ax1.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='gray', alpha=0.5)
                
                im1 = ax1.pcolormesh(lon_coords, lat_coords, target, 
                                    transform=ccrs.PlateCarree(), 
                                    cmap=cmap, vmin=0, vmax=vmax)
                ax1.set_title(f'Sample {sample_count + 1}: Ground Truth', 
                             fontsize=12, fontweight='bold')
                plt.colorbar(im1, ax=ax1, orientation='horizontal', pad=0.05, shrink=0.7)
                
                # Prediction
                ax2 = plt.subplot(n_samples, 3, sample_count * 3 + 2, 
                                 projection=ccrs.PlateCarree())
                ax2.set_extent(india_extent, crs=ccrs.PlateCarree())
                ax2.add_feature(cfeature.COASTLINE, linewidth=0.6)
                ax2.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor='black')
                ax2.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='gray', alpha=0.5)
                
                im2 = ax2.pcolormesh(lon_coords, lat_coords, pred, 
                                    transform=ccrs.PlateCarree(), 
                                    cmap=cmap, vmin=0, vmax=vmax)
                ax2.set_title(f'Prediction', fontsize=12, fontweight='bold')
                plt.colorbar(im2, ax=ax2, orientation='horizontal', pad=0.05, shrink=0.7)
                
                # Error
                ax3 = plt.subplot(n_samples, 3, sample_count * 3 + 3, 
                                 projection=ccrs.PlateCarree())
                ax3.set_extent(india_extent, crs=ccrs.PlateCarree())
                ax3.add_feature(cfeature.COASTLINE, linewidth=0.6)
                ax3.add_feature(cfeature.BORDERS, linewidth=0.6, edgecolor='black')
                ax3.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='gray', alpha=0.5)
                
                error = np.abs(pred - target)
                im3 = ax3.pcolormesh(lon_coords, lat_coords, error, 
                                    transform=ccrs.PlateCarree(), 
                                    cmap='YlOrRd', vmin=0, vmax=error.max())
                mae = np.mean(error)
                ax3.set_title(f'Error (MAE: {mae:.3f} mm)', fontsize=12, fontweight='bold')
                plt.colorbar(im3, ax=ax3, orientation='horizontal', pad=0.05, shrink=0.7)
                
                sample_count += 1
    
    plt.suptitle('Multiple Rainfall Predictions Over India', 
                 fontsize=18, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"🗺️  Multiple samples map saved to {save_path}")
    plt.close()

# ============================================================================
# STEP 8: Train with visualization
# ============================================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n🖥️  Using device: {device}")

model = RainfallUNet(in_channels=6, base_filters=64).to(device)
criterion = BernoulliGammaLoss(occurrence_weight=1.0, intensity_weight=1.0, extreme_weight=3.0)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
scaler = torch.amp.GradScaler('cuda')

total_params = sum(p.numel() for p in model.parameters())
print(f"📊 Total parameters: {total_params:,}")

# History tracking
history = defaultdict(list)

n_epochs = 100
best_r2 = -np.inf

print("\n" + "="*70)
print("TRAINING: U-Net with Bernoulli-Gamma Loss (WORKING BASELINE)")
print("="*70)

for epoch in range(n_epochs):
    train_loss, train_bce, train_gamma = train_epoch(
        model, train_loader, criterion, optimizer, device, scaler
    )
    
    # Store training metrics
    history['train_loss'].append(train_loss)
    history['train_bce'].append(train_bce)
    history['train_gamma'].append(train_gamma)
    
    if epoch % 5 == 0:
        metrics = validate_with_metrics(model, val_loader, device, n_samples=5)
        
        # Store validation metrics
        history['val_mae'].append(metrics['mae'])
        history['val_rmse'].append(metrics['rmse'])
        history['val_r2'].append(metrics['r2'])
        history['val_crps'].append(metrics['crps'])
        
        scheduler.step(metrics['mae'])
        
        if metrics['r2'] > best_r2:
            best_r2 = metrics['r2']
            torch.save(model.state_dict(), 'best_unet_rainfall.pt')
            print(f"Epoch {epoch:3d} | ⭐ New best R²: {metrics['r2']:.4f}")
        
        print(f"Epoch {epoch:3d} | Train: {train_loss:.4f} (BCE: {train_bce:.4f}, Γ: {train_gamma:.4f}) | "
              f"MAE: {metrics['mae']:.4f} | R²: {metrics['r2']:.4f} | CRPS: {metrics['crps']:.4f}")
        
        # Plot training metrics and CRPS maps every 10 epochs
        if epoch % 10 == 0 and epoch > 0:
            print(f"   📊 Generating visualizations for epoch {epoch}...")
            plot_training_metrics(history, save_path=f'training_metrics_v2_epoch{epoch}.png')
            plot_india_rainfall_prediction(model, val_loader, device, lat_coords, lon_coords, 
                                         sample_idx=0, save_path=f'india_crps_v2_map_epoch{epoch}.png')
    else:
        print(f"Epoch {epoch:3d} | Train: {train_loss:.4f}")

print(f"\n✅ Best R²: {best_r2:.4f}")

# Final training metrics plot
plot_training_metrics(history, save_path='training_metrics_v2_final.png')

# Generate final CRPS map with best model
print("\n📊 Generating final CRPS visualization with best model...")
model.load_state_dict(torch.load('best_unet_rainfall.pt'))
plot_india_rainfall_prediction(model, val_loader, device, lat_coords, lon_coords, 
                               sample_idx=0, save_path='india_crps_v2_map_final.png')

# ============================================================================
# STEP 9: Detailed Evaluation with Visualization
# ============================================================================

def detailed_evaluation(model, loader, device):
    print("\n" + "="*70)
    print("DETAILED EVALUATION")
    print("="*70)
    
    metrics = validate_with_metrics(model, loader, device, n_samples=10)
    
    print(f"\n📊 Overall Metrics:")
    print(f"   MAE:  {metrics['mae']:.4f} mm/3h")
    print(f"   RMSE: {metrics['rmse']:.4f} mm/3h")
    print(f"   R²:   {metrics['r2']:.4f}")
    print(f"   CRPS: {metrics['crps']:.4f}")
    
    # Stratified
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            pred = model.predict_rainfall(X_batch)
            all_preds.append(pred.cpu().numpy())
            all_targets.append(y_batch.numpy())
    
    preds = np.concatenate(all_preds, axis=0).flatten()
    targets = np.concatenate(all_targets, axis=0).flatten()
    
    print(f"\n📈 Performance by intensity:")
    for p in [50, 75, 90, 95, 99]:
        threshold = np.percentile(targets[targets > 0], p)
        mask = targets > threshold
        if mask.sum() > 10:
            r2_s = 1 - (np.sum((targets[mask] - preds[mask]) ** 2) / 
                        np.sum((targets[mask] - targets[mask].mean()) ** 2))
            mae_s = np.mean(np.abs(targets[mask] - preds[mask]))
            print(f"   >{p:2d}%ile (>{threshold:6.2f}mm): R²={r2_s:7.4f}, MAE={mae_s:6.4f}, N={mask.sum():,}")

model.load_state_dict(torch.load('best_unet_rainfall.pt'))
detailed_evaluation(model, val_loader, device)

# Generate visualization plots
print("\n" + "="*70)
print("GENERATING VISUALIZATIONS")
print("="*70)

plot_india_rainfall_prediction(model, val_loader, device, lat_coords, lon_coords, 
                               sample_idx=0, save_path='india_rainfall_prediction.png')

plot_multiple_samples_india(model, val_loader, device, lat_coords, lon_coords, 
                            n_samples=4, save_path='india_rainfall_samples.png')

print("\n✅ All visualizations complete!")

Downsampling 0.1° rainfall to 0.5° for input...
Original 0.1° rainfall shape: (2928, 320, 300)
Downsampled 0.5° rainfall shape: (2928, 64, 60)
ERA5 0.5° shape: (2928, 64, 60)
Extreme events (>3.12mm): 2915 / 2928
   Detected coordinates: lat, lon

✅ Normalization complete

🖥️  Using device: cuda
📊 Total parameters: 7,071,555

TRAINING: U-Net with Bernoulli-Gamma Loss (WORKING BASELINE)
Epoch   0 | ⭐ New best R²: -1.2131
Epoch   0 | Train: 1.4159 (BCE: 0.2413, Γ: 1.1745) | MAE: 0.2921 | R²: -1.2131 | CRPS: 0.2682
Epoch   1 | Train: 0.9856
Epoch   2 | Train: 0.8028
Epoch   3 | Train: 0.7043
Epoch   4 | Train: 0.6521
Epoch   5 | ⭐ New best R²: 0.8183
Epoch   5 | Train: 0.6580 (BCE: 0.1201, Γ: 0.5379) | MAE: 0.1034 | R²: 0.8183 | CRPS: 0.0896
Epoch   6 | Train: 0.6530
Epoch   7 | Train: 0.6347
Epoch   8 | Train: 0.6160
Epoch   9 | Train: 0.6004
Epoch  10 | ⭐ New best R²: 0.9188
Epoch  10 | Train: 0.5858 (BCE: 0.1145, Γ: 0.4713) | MAE: 0.0649 | R²: 0.9188 | CRPS: 0.0556
   📊 Generating visu

/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_0_boundary_lines_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_cultural/ne_50m_admin_1_states_provinces_lakes.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


🗺️  India rainfall map saved to india_crps_v2_map_epoch10.png
Epoch  11 | Train: 0.6051
Epoch  12 | Train: 0.5713
Epoch  13 | Train: 0.5758
Epoch  14 | Train: 0.5712
Epoch  15 | Train: 0.5702 (BCE: 0.1117, Γ: 0.4584) | MAE: 0.0903 | R²: 0.8372 | CRPS: 0.0777
Epoch  16 | Train: 0.5469
Epoch  17 | Train: 0.5462
Epoch  18 | Train: 0.5490
Epoch  19 | Train: 0.5189
Epoch  20 | ⭐ New best R²: 0.9201
Epoch  20 | Train: 0.5329 (BCE: 0.1095, Γ: 0.4234) | MAE: 0.0627 | R²: 0.9201 | CRPS: 0.0542
   📊 Generating visualizations for epoch 20...
📊 Training metrics plot saved to training_metrics_v2_epoch20.png
🗺️  India rainfall map saved to india_crps_v2_map_epoch20.png
Epoch  21 | Train: 0.5344
Epoch  22 | Train: 0.5414
Epoch  23 | Train: 0.5287
Epoch  24 | Train: 0.5067
Epoch  25 | ⭐ New best R²: 0.9251
Epoch  25 | Train: 0.5022 (BCE: 0.1061, Γ: 0.3962) | MAE: 0.0602 | R²: 0.9251 | CRPS: 0.0519
Epoch  26 | Train: 0.4707
Epoch  27 | Train: 0.4746
Epoch  28 | Train: 0.4898
Epoch  29 | Train: 0.4933
E

In [7]:
# ============================================================================
# STEP 10: Scatter Plots & R² — All-India and State-wise
# ============================================================================

import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# --------------------------------------------------------------------------
# 10a. Build state-pixel lookup (run once, reuse for all samples)
# --------------------------------------------------------------------------

def build_state_pixel_lookup(lat_coords, lon_coords, shapefile_path=None):
    """
    Assigns each (lat, lon) pixel to an Indian state.
    Falls back to a coarse bounding-box lookup if no shapefile is available.
    Returns: dict  {state_name: flat_pixel_indices}
    """
    print("Building state-pixel lookup table...")

    if shapefile_path is not None:
        # --- Preferred: use a shapefile (e.g. from Natural Earth or GADM) ---
        india_states = gpd.read_file(shapefile_path)
        # Reproject to WGS84 if necessary
        india_states = india_states.to_crs("EPSG:4326")
        # Column that holds state names — adjust if your shapefile uses a different field
        name_col = [c for c in india_states.columns
                    if c.lower() in ('name', 'state_name', 'st_nm', 'shapename')][0]
    else:
        # --- Fallback: approximate bounding boxes for major states ---
        print("   ⚠️  No shapefile provided — using approximate bounding boxes.")
        india_states = None

    H, W = len(lat_coords), len(lon_coords)
    state_pixel_map = {}   # state_name -> list of flat indices

    # Build a meshgrid of all pixel centres
    lons_2d, lats_2d = np.meshgrid(lon_coords, lat_coords)   # (H, W)
    flat_lats = lats_2d.flatten()
    flat_lons = lons_2d.flatten()

    if india_states is not None:
        # Batch spatial join using GeoDataFrame
        points_gdf = gpd.GeoDataFrame(
            {'idx': np.arange(len(flat_lats))},
            geometry=[Point(lo, la) for lo, la in zip(flat_lons, flat_lats)],
            crs="EPSG:4326"
        )
        joined = gpd.sjoin(points_gdf, india_states[[name_col, 'geometry']],
                           how='left', predicate='within')
        joined[name_col] = joined[name_col].fillna('Ocean/Outside')

        for state, grp in joined.groupby(name_col):
            if state != 'Ocean/Outside':
                state_pixel_map[state] = grp['idx'].tolist()
    else:
        # Coarse bounding-box fallback
        STATE_BOXES = {
            'Rajasthan':       (23.0, 30.5, 69.5, 78.5),
            'Madhya Pradesh':  (21.0, 26.5, 74.0, 82.5),
            'Maharashtra':     (15.5, 22.5, 72.5, 80.5),
            'Uttar Pradesh':   (23.5, 30.5, 77.0, 84.5),
            'Gujarat':         (20.0, 24.5, 68.0, 74.5),
            'Karnataka':       (11.5, 18.5, 74.0, 78.5),
            'Andhra Pradesh':  (12.5, 19.5, 76.5, 84.5),
            'Tamil Nadu':      ( 8.0, 13.5, 76.5, 80.5),
            'West Bengal':     (21.5, 27.5, 85.5, 89.5),
            'Odisha':          (17.5, 22.5, 81.5, 87.5),
            'Bihar':           (24.0, 28.0, 83.5, 88.5),
            'Punjab':          (29.5, 32.5, 73.5, 76.5),
            'Haryana':         (27.5, 30.5, 74.5, 77.5),
            'Jharkhand':       (21.5, 25.0, 83.5, 87.5),
            'Chhattisgarh':    (17.5, 24.0, 80.0, 84.5),
            'Telangana':       (15.5, 19.5, 77.0, 81.5),
            'Kerala':          ( 8.0, 12.5, 74.5, 77.5),
            'Himachal Pradesh':(30.5, 33.0, 75.5, 79.0),
            'Uttarakhand':     (28.5, 31.5, 78.0, 81.5),
            'Assam':           (24.0, 28.0, 89.5, 96.5),
            'Nagaland':        (25.0, 27.5, 93.5, 95.5),
            'Manipur':         (23.5, 25.5, 93.0, 95.0),
            'Meghalaya':       (24.5, 26.5, 89.5, 92.5),
            'Tripura':         (22.5, 24.5, 91.0, 92.5),
            'Mizoram':         (21.5, 24.5, 92.0, 93.5),
            'Goa':             (14.9, 15.8, 73.8, 74.4),
            'J&K / Ladakh':    (32.0, 37.5, 73.5, 80.5),
            'Arunachal Pradesh':(26.5, 29.5, 91.5, 97.5),
        }
        for state, (lat_min, lat_max, lon_min, lon_max) in STATE_BOXES.items():
            mask = ((flat_lats >= lat_min) & (flat_lats <= lat_max) &
                    (flat_lons >= lon_min) & (flat_lons <= lon_max))
            idxs = np.where(mask)[0].tolist()
            if len(idxs) > 50:          # skip negligibly small regions
                state_pixel_map[state] = idxs

    print(f"   Found {len(state_pixel_map)} states/regions")
    return state_pixel_map


# --------------------------------------------------------------------------
# 10b. Collect all-validation predictions at 0.1° resolution
# --------------------------------------------------------------------------

def collect_all_predictions(model, loader, device):
    """Returns flat arrays of (predictions, targets) over the full val set."""
    model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            pred = model.predict_rainfall(X_batch)      # (B, H, W)
            all_preds.append(pred.cpu().numpy())
            all_targets.append(y_batch.numpy())
    preds   = np.concatenate(all_preds,   axis=0)       # (N, H, W)
    targets = np.concatenate(all_targets, axis=0)       # (N, H, W)
    return preds, targets


# --------------------------------------------------------------------------
# 10c. Scatter + stats helper
# --------------------------------------------------------------------------

def _scatter_stats(ax, obs, pred, title, color='#2196F3',
                   max_pts=5000, annotation_loc='upper left'):
    """Draw a scatter panel with 1:1 line, best-fit line, and stats box."""
    # Thin to a manageable number of points
    if len(obs) > max_pts:
        idx = np.random.choice(len(obs), max_pts, replace=False)
        obs_plot, pred_plot = obs[idx], pred[idx]
    else:
        obs_plot, pred_plot = obs, pred

    ax.scatter(obs_plot, pred_plot, alpha=0.25, s=6, color=color,
               edgecolors='none', rasterized=True)

    # 1:1 reference line
    vmax = max(obs.max(), pred.max()) * 1.05
    ax.plot([0, vmax], [0, vmax], 'k--', lw=1.2, label='1:1 line')

    # Best-fit line (on full data)
    valid = np.isfinite(obs) & np.isfinite(pred)
    if valid.sum() > 5:
        m, b = np.polyfit(obs[valid], pred[valid], 1)
        x_fit = np.array([0, vmax])
        ax.plot(x_fit, m * x_fit + b, 'r-', lw=1.2,
                label=f'fit  y={m:.2f}x+{b:.2f}')

    # Metrics
    ss_res = np.sum((obs[valid] - pred[valid]) ** 2)
    ss_tot = np.sum((obs[valid] - obs[valid].mean()) ** 2)
    r2   = 1 - ss_res / (ss_tot + 1e-12)
    mae  = np.mean(np.abs(obs[valid] - pred[valid]))
    rmse = np.sqrt(np.mean((obs[valid] - pred[valid]) ** 2))
    bias = np.mean(pred[valid] - obs[valid])
    corr = np.corrcoef(obs[valid], pred[valid])[0, 1]

    stats_txt = (f"R² = {r2:.3f}\n"
                 f"r  = {corr:.3f}\n"
                 f"MAE = {mae:.3f} mm\n"
                 f"RMSE = {rmse:.3f} mm\n"
                 f"Bias = {bias:+.3f} mm\n"
                 f"N = {valid.sum():,}")

    loc_kwargs = dict(upper_left  = dict(x=0.04, y=0.96, ha='left',  va='top'),
                      upper_right = dict(x=0.96, y=0.96, ha='right', va='top'),
                      lower_right = dict(x=0.96, y=0.04, ha='right', va='bottom'))
    lk = loc_kwargs.get(annotation_loc, loc_kwargs['upper_left'])

    ax.text(lk['x'], lk['y'], stats_txt,
            transform=ax.transAxes, fontsize=8.5,
            ha=lk['ha'], va=lk['va'],
            bbox=dict(boxstyle='round,pad=0.4', fc='white',
                      ec='#cccccc', alpha=0.85))

    ax.set_xlim(0, vmax);  ax.set_ylim(0, vmax)
    ax.set_xlabel('Observed (mm / 3h)', fontsize=9)
    ax.set_ylabel('Predicted (mm / 3h)', fontsize=9)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_aspect('equal', adjustable='box')
    ax.grid(True, alpha=0.25, lw=0.5)

    return {'r2': r2, 'corr': corr, 'mae': mae, 'rmse': rmse,
            'bias': bias, 'n': int(valid.sum())}


# --------------------------------------------------------------------------
# 10d. All-India scatter (full 0.1° grid)
# --------------------------------------------------------------------------

def plot_all_india_scatter(preds_flat, targets_flat,
                           save_path='scatter_all_india.png'):
    """
    Single scatter panel — all pixels, all validation time steps, all-India.
    Also breaks down performance for dry / light / moderate / heavy rain.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('All-India downscaling performance — 0.1° resolution',
                 fontsize=14, fontweight='bold')

    # ---- Panel 1: all pixels (log scale) ----
    _scatter_stats(axes[0], targets_flat, preds_flat,
                   'All pixels (log scale)', color='#1565C0')
    axes[0].set_xscale('symlog', linthresh=0.1)
    axes[0].set_yscale('symlog', linthresh=0.1)

    # ---- Panel 2: rainy pixels only (> 0.1 mm) ----
    rain_mask = targets_flat > 0.1
    _scatter_stats(axes[1],
                   targets_flat[rain_mask], preds_flat[rain_mask],
                   'Rainy pixels only (> 0.1 mm)', color='#0277BD')

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅  All-India scatter saved → {save_path}")
    plt.close()

    # ---- Intensity-stratified stats (printed) ----
    print("\n📊 All-India: performance by intensity bin")
    print(f"{'Bin':<22}  {'R²':>7}  {'r':>6}  {'MAE':>7}  {'RMSE':>7}  {'Bias':>8}  {'N':>8}")
    bins = [
        ('Dry  (0 mm)',        targets_flat == 0),
        ('Trace (0–1 mm)',     (targets_flat > 0)   & (targets_flat <= 1)),
        ('Light (1–5 mm)',     (targets_flat > 1)   & (targets_flat <= 5)),
        ('Moderate (5–20 mm)',(targets_flat > 5)   & (targets_flat <= 20)),
        ('Heavy (20–50 mm)',   (targets_flat > 20)  & (targets_flat <= 50)),
        ('Extreme (>50 mm)',   targets_flat > 50),
    ]
    for label, mask in bins:
        if mask.sum() < 10:
            continue
        o, p = targets_flat[mask], preds_flat[mask]
        ss_res = np.sum((o - p) ** 2)
        ss_tot = np.sum((o - o.mean()) ** 2)
        r2   = 1 - ss_res / (ss_tot + 1e-12)
        corr = np.corrcoef(o, p)[0, 1] if len(o) > 2 else np.nan
        mae  = np.mean(np.abs(o - p))
        rmse = np.sqrt(np.mean((o - p) ** 2))
        bias = np.mean(p - o)
        print(f"{label:<22}  {r2:>7.3f}  {corr:>6.3f}  {mae:>7.3f}  "
              f"{rmse:>7.3f}  {bias:>+8.3f}  {mask.sum():>8,}")


# --------------------------------------------------------------------------
# 10e. State-wise scatter (grid of panels)
# --------------------------------------------------------------------------

def plot_statewise_scatter(preds_flat_full, targets_flat_full,
                           state_pixel_map, n_samples_total,
                           spatial_shape,
                           save_path='scatter_statewise.png'):
    """
    One scatter panel per state.  preds/targets are (N*H*W,) flat arrays
    where N = number of validation time steps, spatial_shape = (H, W).
    """
    states = sorted(state_pixel_map.keys())
    n_states = len(states)
    ncols = 4
    nrows = (n_states + ncols - 1) // ncols

    # Pastel palette — cycles if more than 20 states
    COLORS = plt.cm.tab20.colors

    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 4.2, nrows * 4.2))
    axes_flat = axes.flatten()

    # Build per-pixel index for time-stacked array
    H, W = spatial_shape
    state_stats = {}

    for ax_idx, state in enumerate(states):
        ax = axes_flat[ax_idx]
        pix_idxs = state_pixel_map[state]   # spatial flat indices (0..H*W-1)

        # For each time step the spatial flat index repeats
        # Full flat array layout: (n_samples * H * W,)
        # spatial pixel k at time step t → index t*H*W + k
        all_t_idxs = np.concatenate(
            [np.array(pix_idxs) + t * H * W
             for t in range(n_samples_total)]
        )
        # Clip to valid range (safety)
        all_t_idxs = all_t_idxs[all_t_idxs < len(preds_flat_full)]

        o_state = targets_flat_full[all_t_idxs]
        p_state = preds_flat_full[all_t_idxs]

        color = COLORS[ax_idx % len(COLORS)]
        stats = _scatter_stats(ax, o_state, p_state,
                               state, color=color,
                               max_pts=3000,
                               annotation_loc='upper left')
        state_stats[state] = stats

    # Hide unused axes
    for ax_idx in range(n_states, len(axes_flat)):
        axes_flat[ax_idx].set_visible(False)

    fig.suptitle('State-wise downscaling performance — 0.1° resolution',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    print(f"✅  State-wise scatter saved → {save_path}")
    plt.close()

    return state_stats


# --------------------------------------------------------------------------
# 10f. State R² summary table + bar chart
# --------------------------------------------------------------------------

def plot_state_r2_summary(state_stats, save_path='state_r2_summary.png'):
    """Bar chart + table of R², MAE, RMSE for every state, sorted by R²."""
    df = pd.DataFrame(state_stats).T.reset_index()
    df.columns = ['State', 'R²', 'r', 'MAE', 'RMSE', 'Bias', 'N']
    df = df.sort_values('R²', ascending=True).reset_index(drop=True)

    fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(df) * 0.38 + 2)),
                             gridspec_kw={'width_ratios': [2, 1]})

    # ---- Left: horizontal bar chart ----
    colors = plt.cm.RdYlGn(
        (df['R²'].values - df['R²'].min()) /
        (df['R²'].max() - df['R²'].min() + 1e-9)
    )
    bars = axes[0].barh(df['State'], df['R²'],
                        color=colors, edgecolor='white', linewidth=0.5)
    axes[0].axvline(0.0, color='red',   lw=1.0, ls='--', alpha=0.7)
    axes[0].axvline(0.5, color='orange',lw=0.8, ls='--', alpha=0.5)
    axes[0].axvline(0.7, color='green', lw=0.8, ls='--', alpha=0.5)
    for bar, (_, row) in zip(bars, df.iterrows()):
        axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                     f"{row['R²']:.3f}", va='center', ha='left', fontsize=8)
    axes[0].set_xlim(-0.1, 1.05)
    axes[0].set_xlabel('R²', fontsize=11)
    axes[0].set_title('R² by state (sorted)', fontsize=12, fontweight='bold')
    axes[0].grid(axis='x', alpha=0.3)

    # ---- Right: numeric table ----
    axes[1].axis('off')
    tbl_data = df[['State', 'R²', 'MAE', 'RMSE', 'Bias']].copy()
    tbl_data['R²']   = tbl_data['R²'].map('{:.3f}'.format)
    tbl_data['MAE']  = tbl_data['MAE'].map('{:.3f}'.format)
    tbl_data['RMSE'] = tbl_data['RMSE'].map('{:.3f}'.format)
    tbl_data['Bias'] = tbl_data['Bias'].map('{:+.3f}'.format)
    tbl = axes[1].table(
        cellText=tbl_data.values,
        colLabels=tbl_data.columns,
        cellLoc='center', loc='center',
        bbox=[0, 0, 1, 1]
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8)
    for (r, c), cell in tbl.get_celld().items():
        if r == 0:
            cell.set_facecolor('#333333')
            cell.set_text_props(color='white', fontweight='bold')
        elif r % 2 == 0:
            cell.set_facecolor('#f5f5f5')
    axes[1].set_title('Numeric summary', fontsize=12, fontweight='bold', pad=4)

    fig.suptitle('All-state R² / error summary — 0.1° downscaling',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    print(f"✅  State R² summary saved → {save_path}")
    plt.close()

    # Also print to console
    print("\n📋 State-wise statistics (sorted by R²):")
    print(df.to_string(index=False, float_format='{:.4f}'.format))
    return df


# --------------------------------------------------------------------------
# 10g.  Optional — spatial R² choropleth on India map
# --------------------------------------------------------------------------

def plot_r2_india_choropleth(state_stats, lat_coords, lon_coords,
                              save_path='r2_choropleth_india.png'):
    """
    Colour each grid cell by the R² of the state it belongs to.
    Requires cartopy (already imported).
    Uses the same bounding-box approximation as the lookup table,
    so it works without a shapefile.
    """
    H, W = len(lat_coords), len(lon_coords)
    r2_map = np.full((H, W), np.nan)

    # Re-build per-pixel assignment from bounding boxes
    lons_2d, lats_2d = np.meshgrid(lon_coords, lat_coords)
    STATE_BOXES = {
        'Rajasthan':       (23.0, 30.5, 69.5, 78.5),
        'Madhya Pradesh':  (21.0, 26.5, 74.0, 82.5),
        'Maharashtra':     (15.5, 22.5, 72.5, 80.5),
        'Uttar Pradesh':   (23.5, 30.5, 77.0, 84.5),
        'Gujarat':         (20.0, 24.5, 68.0, 74.5),
        'Karnataka':       (11.5, 18.5, 74.0, 78.5),
        'Andhra Pradesh':  (12.5, 19.5, 76.5, 84.5),
        'Tamil Nadu':      ( 8.0, 13.5, 76.5, 80.5),
        'West Bengal':     (21.5, 27.5, 85.5, 89.5),
        'Odisha':          (17.5, 22.5, 81.5, 87.5),
        'Bihar':           (24.0, 28.0, 83.5, 88.5),
        'Punjab':          (29.5, 32.5, 73.5, 76.5),
        'Haryana':         (27.5, 30.5, 74.5, 77.5),
        'Jharkhand':       (21.5, 25.0, 83.5, 87.5),
        'Chhattisgarh':    (17.5, 24.0, 80.0, 84.5),
        'Telangana':       (15.5, 19.5, 77.0, 81.5),
        'Kerala':          ( 8.0, 12.5, 74.5, 77.5),
        'Himachal Pradesh':(30.5, 33.0, 75.5, 79.0),
        'Uttarakhand':     (28.5, 31.5, 78.0, 81.5),
        'Assam':           (24.0, 28.0, 89.5, 96.5),
        'Goa':             (14.9, 15.8, 73.8, 74.4),
        'J&K / Ladakh':    (32.0, 37.5, 73.5, 80.5),
        'Arunachal Pradesh':(26.5, 29.5, 91.5, 97.5),
    }
    for state, (lat_min, lat_max, lon_min, lon_max) in STATE_BOXES.items():
        if state not in state_stats:
            continue
        r2_val = state_stats[state]['r2']
        mask = ((lats_2d >= lat_min) & (lats_2d <= lat_max) &
                (lons_2d >= lon_min) & (lons_2d <= lon_max))
        r2_map[mask] = r2_val

    # Plot
    fig = plt.figure(figsize=(10, 10))
    ax = plt.axes(projection=ccrs.PlateCarree())
    india_extent = [lon_coords.min() - 0.5, lon_coords.max() + 0.5,
                    lat_coords.min() - 0.5, lat_coords.max() + 0.5]
    ax.set_extent(india_extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS, linewidth=0.8, edgecolor='black')
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray')

    cmap_r2 = plt.cm.RdYlGn
    cmap_r2.set_bad('lightgray')
    im = ax.pcolormesh(lon_coords, lat_coords, r2_map,
                       transform=ccrs.PlateCarree(),
                       cmap=cmap_r2, vmin=0, vmax=1)
    cbar = plt.colorbar(im, ax=ax, orientation='vertical',
                        pad=0.04, shrink=0.6)
    cbar.set_label('R²', fontsize=12)

    # Annotate states with R² value
    for state, (lat_min, lat_max, lon_min, lon_max) in STATE_BOXES.items():
        if state not in state_stats:
            continue
        cx = (lon_min + lon_max) / 2
        cy = (lat_min + lat_max) / 2
        r2_val = state_stats[state]['r2']
        ax.text(cx, cy, f'{r2_val:.2f}',
                transform=ccrs.PlateCarree(),
                ha='center', va='center', fontsize=6.5,
                fontweight='bold', color='black',
                bbox=dict(fc='white', ec='none', alpha=0.6, pad=1))

    ax.set_title('State-wise R² — 0.1° downscaling performance',
                 fontsize=13, fontweight='bold', pad=15)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✅  R² choropleth saved → {save_path}")
    plt.close()


# --------------------------------------------------------------------------
# 10h. RUN EVERYTHING
# --------------------------------------------------------------------------

print("\n" + "="*70)
print("STEP 10 — SCATTER PLOTS & STATE-WISE R² ANALYSIS")
print("="*70)

# ── 1. Collect predictions ──────────────────────────────────────────────
model.load_state_dict(torch.load('best_unet_rainfall.pt'))
model.eval()

print("\n[1/5] Collecting validation predictions...")
preds_all, targets_all = collect_all_predictions(model, val_loader, device)
N, H, W = preds_all.shape
print(f"      Collected {N} time steps × {H}×{W} grid = {N*H*W:,} pixels")

preds_flat   = preds_all.flatten()
targets_flat = targets_all.flatten()

# ── 2. All-India scatter ────────────────────────────────────────────────
print("\n[2/5] All-India scatter plots...")
plot_all_india_scatter(preds_flat, targets_flat,
                       save_path='scatter_all_india.png')

# ── 3. Build state pixel lookup ──────────────────────────────────────────
print("\n[3/5] Building state-pixel lookup...")
# To use a proper shapefile (strongly recommended):
#   state_pixel_map = build_state_pixel_lookup(lat_coords, lon_coords,
#                         shapefile_path='/path/to/india_states.shp')
# Fallback (bounding boxes):
state_pixel_map = build_state_pixel_lookup(lat_coords, lon_coords,
                                            shapefile_path=None)

# ── 4. State-wise scatter ───────────────────────────────────────────────
print("\n[4/5] State-wise scatter plots...")
state_stats = plot_statewise_scatter(
    preds_flat, targets_flat,
    state_pixel_map,
    n_samples_total=N,
    spatial_shape=(H, W),
    save_path='scatter_statewise.png'
)

# ── 5. Summary bar chart + table + choropleth ──────────────────────────
print("\n[5/5] R² summary & choropleth...")
summary_df = plot_state_r2_summary(state_stats,
                                    save_path='state_r2_summary.png')
plot_r2_india_choropleth(state_stats, lat_coords, lon_coords,
                          save_path='r2_choropleth_india.png')

print("\n" + "="*70)
print("ALL OUTPUT FILES:")
print("  scatter_all_india.png   — all-pixel + rainy-pixel scatter, India")
print("  scatter_statewise.png   — one panel per state")
print("  state_r2_summary.png    — sorted bar chart + table")
print("  r2_choropleth_india.png — R² painted onto the India map")
print("="*70)


STEP 10 — SCATTER PLOTS & STATE-WISE R² ANALYSIS

[1/5] Collecting validation predictions...
      Collected 586 time steps × 320×300 grid = 56,256,000 pixels

[2/5] All-India scatter plots...
✅  All-India scatter saved → scatter_all_india.png

📊 All-India: performance by intensity bin
Bin                          R²       r      MAE     RMSE      Bias         N


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Dry  (0 mm)             -23698577272340480.000     nan    0.002    0.024    +0.002  41,076,862
Trace (0–1 mm)            0.445   0.819    0.086    0.180    -0.004  11,456,295
Light (1–5 mm)            0.685   0.863    0.386    0.599    +0.015  3,113,513
Moderate (5–20 mm)        0.630   0.836    1.064    1.683    -0.415   595,891
Heavy (20–50 mm)         -0.664   0.520    6.461    8.567    -4.811    12,907
Extreme (>50 mm)         -4.602   0.252   24.195   28.230   -23.779       532

[3/5] Building state-pixel lookup...
Building state-pixel lookup table...
   ⚠️  No shapefile provided — using approximate bounding boxes.
   Found 28 states/regions

[4/5] State-wise scatter plots...
✅  State-wise scatter saved → scatter_statewise.png

[5/5] R² summary & choropleth...
✅  State R² summary saved → state_r2_summary.png

📋 State-wise statistics (sorted by R²):
            State     R²      r    MAE   RMSE    Bias            N
              Goa 0.8422 0.9187 0.0596 0.3427  0.0038   31644.0000


In [8]:
# ============================================================================
# STEP 11: Side-by-side rainfall visualization — 0.5° input vs 0.1° output
#          10 validation samples with ground truth comparison
# ============================================================================

def plot_05_vs_01_comparison(model, val_loader, device,
                              lat_coords_01, lon_coords_01,
                              era5_india_05,
                              rain_05_downsampled,
                              n_samples=10,
                              save_path='rainfall_05_vs_01_comparison.png'):
    """
    For each sample, shows 4 panels:
        Col 1 — 0.5° input rainfall  (coarse, downsampled from 0.1°)
        Col 2 — 0.1° ground truth    (fine, target)
        Col 3 — 0.1° model prediction (fine, output)
        Col 4 — Absolute error        (prediction − truth)

    Lat/lon grids are derived automatically from the ERA5 0.5° object
    and the original 0.1° coordinate arrays.
    """

    # ── Coordinate arrays ──────────────────────────────────────────────────
    # 0.5° grid (from coarsened ERA5)
    lat_05 = era5_india_05['u10'].coords[
        [c for c in era5_india_05['u10'].coords
         if c in ('latitude', 'lat', 'y')][0]].values
    lon_05 = era5_india_05['u10'].coords[
        [c for c in era5_india_05['u10'].coords
         if c in ('longitude', 'lon', 'x')][0]].values

    # 0.1° grid (original)
    lat_01 = lat_coords_01
    lon_01 = lon_coords_01

    india_extent_01 = [lon_01.min(), lon_01.max(),
                       lat_01.min(), lat_01.max()]
    india_extent_05 = [lon_05.min(), lon_05.max(),
                       lat_05.min(), lat_05.max()]

    # ── Colourmap ──────────────────────────────────────────────────────────
    rain_colors = [
        '#FFFFFF', '#E0F3DB', '#A8DDB5', '#7BCCC4',
        '#4EB3D3', '#2B8CBE', '#0868AC', '#084081',
        '#810F7C', '#8B0000',
    ]
    rain_cmap = LinearSegmentedColormap.from_list(
        'rainfall', rain_colors, N=256)

    # ── Collect model predictions ──────────────────────────────────────────
    model.eval()
    collected = []     # list of (rain_05_frame, target_01, pred_01, t_idx)

    # val_loader indices start at train_size in the original dataset
    val_start_idx = int(0.8 * len(rain_05_downsampled))

    sample_count = 0
    with torch.no_grad():
        for batch_idx, (X_batch, y_batch) in enumerate(val_loader):
            if sample_count >= n_samples:
                break
            X_batch = X_batch.to(device)
            preds = model.predict_rainfall(X_batch)   # (B, H01, W01)

            for i in range(X_batch.shape[0]):
                if sample_count >= n_samples:
                    break
                global_t = val_start_idx + batch_idx * val_loader.batch_size + i

                collected.append({
                    'rain_05': rain_05_downsampled[global_t],   # (H05, W05)
                    'target':  y_batch[i].numpy(),              # (H01, W01)
                    'pred':    preds[i].cpu().numpy(),          # (H01, W01)
                    't_idx':   global_t,
                })
                sample_count += 1

    # ── Figure layout: n_samples rows × 4 columns ─────────────────────────
    ncols = 4
    nrows = n_samples
    col_titles = ['0.5° input (coarse)',
                  '0.1° ground truth (fine)',
                  '0.1° prediction (fine)',
                  'Absolute error']

    fig = plt.figure(figsize=(ncols * 4.5, nrows * 4.0))
    fig.patch.set_facecolor('#1a1a2e')          # dark navy background

    for row, sample in enumerate(collected):
        rain_05 = sample['rain_05']
        target  = sample['target']
        pred    = sample['pred']
        t_idx   = sample['t_idx']
        error   = np.abs(pred - target)

        # Shared colour scale for rain panels (0.5° and 0.1°)
        vmax_rain = max(float(rain_05.max()),
                        float(target.max()),
                        float(pred.max()),
                        0.1)
        vmax_err  = max(float(error.max()), 0.1)

        panels = [
            dict(data=rain_05,  lons=lon_05, lats=lat_05,
                 extent=india_extent_05,
                 cmap=rain_cmap, vmin=0, vmax=vmax_rain,
                 cbar_label='mm / 3h'),
            dict(data=target,   lons=lon_01, lats=lat_01,
                 extent=india_extent_01,
                 cmap=rain_cmap, vmin=0, vmax=vmax_rain,
                 cbar_label='mm / 3h'),
            dict(data=pred,     lons=lon_01, lats=lat_01,
                 extent=india_extent_01,
                 cmap=rain_cmap, vmin=0, vmax=vmax_rain,
                 cbar_label='mm / 3h'),
            dict(data=error,    lons=lon_01, lats=lat_01,
                 extent=india_extent_01,
                 cmap='YlOrRd',  vmin=0, vmax=vmax_err,
                 cbar_label='error (mm)'),
        ]

        for col, panel in enumerate(panels):
            ax_idx = row * ncols + col + 1
            ax = fig.add_subplot(nrows, ncols, ax_idx,
                                 projection=ccrs.PlateCarree())
            ax.set_extent(panel['extent'], crs=ccrs.PlateCarree())
            ax.set_facecolor('#0d0d1a')

            # Map features
            ax.add_feature(cfeature.COASTLINE,
                           linewidth=0.7, edgecolor='#aaaaaa')
            ax.add_feature(cfeature.BORDERS,
                           linewidth=0.7, edgecolor='#cccccc')
            ax.add_feature(cfeature.STATES,
                           linewidth=0.4, edgecolor='#888888', alpha=0.6)
            ax.add_feature(cfeature.OCEAN,
                           facecolor='#0d0d1a', zorder=0)
            ax.add_feature(cfeature.LAND,
                           facecolor='#1c1c2e', zorder=0)

            im = ax.pcolormesh(
                panel['lons'], panel['lats'], panel['data'],
                transform=ccrs.PlateCarree(),
                cmap=panel['cmap'],
                vmin=panel['vmin'], vmax=panel['vmax'],
                shading='auto', zorder=1
            )

            # Colorbar
            cbar = plt.colorbar(im, ax=ax,
                                orientation='horizontal',
                                pad=0.04, shrink=0.85,
                                aspect=22)
            cbar.set_label(panel['cbar_label'],
                           fontsize=7.5, color='#dddddd')
            cbar.ax.tick_params(labelsize=6.5, colors='#aaaaaa')
            cbar.outline.set_edgecolor('#555555')

            # Panel title (only first row)
            if row == 0:
                ax.set_title(col_titles[col],
                             fontsize=10, fontweight='bold',
                             color='#eeeeee', pad=8)

            # Row label (only first column)
            if col == 0:
                # Compute quick stats for this sample
                mae_s  = float(np.mean(np.abs(pred - target)))
                rmse_s = float(np.sqrt(np.mean((pred - target) ** 2)))
                ss_res = np.sum((target - pred) ** 2)
                ss_tot = np.sum((target - target.mean()) ** 2)
                r2_s   = 1.0 - ss_res / (ss_tot + 1e-12)

                ax.text(-0.04, 0.5,
                        f'Sample {row + 1}\n(t={t_idx})\n\n'
                        f'R²={r2_s:.3f}\n'
                        f'MAE={mae_s:.3f}\n'
                        f'RMSE={rmse_s:.3f}',
                        transform=ax.transAxes,
                        fontsize=7.5, color='#cccccc',
                        ha='right', va='center',
                        linespacing=1.6)

            # Resolution badge
            res_label = '0.5°' if col == 0 else '0.1°'
            ax.text(0.02, 0.97, res_label,
                    transform=ax.transAxes,
                    fontsize=7, color='white', fontweight='bold',
                    va='top',
                    bbox=dict(fc='#333366', ec='none',
                              alpha=0.8, pad=2, boxstyle='round'))

    # ── Global title & colour-scale legend explanation ─────────────────────
    fig.suptitle(
        'Rainfall downscaling: 0.5° coarse input  →  0.1° fine prediction\n'
        '(10 validation samples)',
        fontsize=14, fontweight='bold', color='white', y=1.005
    )

    plt.tight_layout(h_pad=0.6, w_pad=0.3)
    plt.savefig(save_path, dpi=200, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    print(f"✅  0.5° vs 0.1° comparison saved → {save_path}")
    plt.close()


# ── Bonus: strip-style summary — mean rainfall maps across all 10 samples ──

def plot_mean_rainfall_strip(model, val_loader, device,
                              lat_coords_01, lon_coords_01,
                              era5_india_05,
                              rain_05_downsampled,
                              n_samples=10,
                              save_path='rainfall_mean_strip.png'):
    """
    3-panel strip showing the *time-mean* over the 10 selected samples:
        Left   — mean 0.5° input
        Centre — mean 0.1° ground truth
        Right  — mean 0.1° prediction
    Plus a difference map (prediction − truth) as a 4th panel.
    """
    lat_05 = era5_india_05['u10'].coords[
        [c for c in era5_india_05['u10'].coords
         if c in ('latitude', 'lat', 'y')][0]].values
    lon_05 = era5_india_05['u10'].coords[
        [c for c in era5_india_05['u10'].coords
         if c in ('longitude', 'lon', 'x')][0]].values
    lat_01, lon_01 = lat_coords_01, lon_coords_01

    val_start_idx = int(0.8 * len(rain_05_downsampled))

    model.eval()
    rain05_acc = []
    target_acc = []
    pred_acc   = []

    sample_count = 0
    with torch.no_grad():
        for batch_idx, (X_batch, y_batch) in enumerate(val_loader):
            if sample_count >= n_samples:
                break
            X_batch = X_batch.to(device)
            preds = model.predict_rainfall(X_batch)

            for i in range(X_batch.shape[0]):
                if sample_count >= n_samples:
                    break
                global_t = val_start_idx + batch_idx * val_loader.batch_size + i
                rain05_acc.append(rain_05_downsampled[global_t])
                target_acc.append(y_batch[i].numpy())
                pred_acc.append(preds[i].cpu().numpy())
                sample_count += 1

    mean_05  = np.mean(rain05_acc, axis=0)
    mean_tgt = np.mean(target_acc, axis=0)
    mean_prd = np.mean(pred_acc,   axis=0)
    mean_diff = mean_prd - mean_tgt          # signed bias map

    rain_colors = [
        '#FFFFFF', '#E0F3DB', '#A8DDB5', '#7BCCC4',
        '#4EB3D3', '#2B8CBE', '#0868AC', '#084081',
        '#810F7C', '#8B0000',
    ]
    rain_cmap = LinearSegmentedColormap.from_list(
        'rainfall', rain_colors, N=256)

    india_ext_01 = [lon_01.min(), lon_01.max(),
                    lat_01.min(), lat_01.max()]
    india_ext_05 = [lon_05.min(), lon_05.max(),
                    lat_05.min(), lat_05.max()]

    vmax = max(mean_05.max(), mean_tgt.max(), mean_prd.max(), 0.1)
    dlim = max(abs(mean_diff.min()), abs(mean_diff.max()), 0.1)

    fig, axes = plt.subplots(1, 4, figsize=(22, 7),
                             subplot_kw={'projection': ccrs.PlateCarree()})
    fig.patch.set_facecolor('#12121f')

    configs = [
        dict(ax=axes[0], data=mean_05,   lons=lon_05, lats=lat_05,
             extent=india_ext_05,
             cmap=rain_cmap, vmin=0, vmax=vmax,
             title='Mean 0.5° input\n(coarse)',
             cbar_label='mm / 3h'),
        dict(ax=axes[1], data=mean_tgt,  lons=lon_01, lats=lat_01,
             extent=india_ext_01,
             cmap=rain_cmap, vmin=0, vmax=vmax,
             title='Mean 0.1° ground truth\n(fine — observed)',
             cbar_label='mm / 3h'),
        dict(ax=axes[2], data=mean_prd,  lons=lon_01, lats=lat_01,
             extent=india_ext_01,
             cmap=rain_cmap, vmin=0, vmax=vmax,
             title='Mean 0.1° prediction\n(fine — model output)',
             cbar_label='mm / 3h'),
        dict(ax=axes[3], data=mean_diff, lons=lon_01, lats=lat_01,
             extent=india_ext_01,
             cmap='RdBu_r',  vmin=-dlim, vmax=dlim,
             title='Mean bias\n(pred − truth)',
             cbar_label='mm / 3h  (signed)'),
    ]

    for cfg in configs:
        ax = cfg['ax']
        ax.set_extent(cfg['extent'], crs=ccrs.PlateCarree())
        ax.set_facecolor('#0d0d1a')
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8, edgecolor='#aaaaaa')
        ax.add_feature(cfeature.BORDERS,   linewidth=0.8, edgecolor='#cccccc')
        ax.add_feature(cfeature.STATES,    linewidth=0.4, edgecolor='#888888')
        ax.add_feature(cfeature.LAND,      facecolor='#1c1c2e', zorder=0)
        ax.add_feature(cfeature.OCEAN,     facecolor='#0d0d1a', zorder=0)

        im = ax.pcolormesh(cfg['lons'], cfg['lats'], cfg['data'],
                           transform=ccrs.PlateCarree(),
                           cmap=cfg['cmap'],
                           vmin=cfg['vmin'], vmax=cfg['vmax'],
                           shading='auto', zorder=1)
        cbar = plt.colorbar(im, ax=ax, orientation='horizontal',
                            pad=0.04, shrink=0.85)
        cbar.set_label(cfg['cbar_label'], fontsize=9, color='#dddddd')
        cbar.ax.tick_params(labelsize=8, colors='#aaaaaa')
        cbar.outline.set_edgecolor('#555555')
        ax.set_title(cfg['title'], fontsize=11, fontweight='bold',
                     color='#eeeeee', pad=10)

        # Resolution watermark
        res = '0.5°' if cfg['ax'] is axes[0] else '0.1°'
        ax.text(0.02, 0.97, res,
                transform=ax.transAxes, fontsize=8,
                color='white', fontweight='bold', va='top',
                bbox=dict(fc='#333366', ec='none', alpha=0.85,
                          pad=2.5, boxstyle='round'))

    # Global stats annotation on bias panel
    mae_mean  = float(np.mean(np.abs(mean_diff)))
    rmse_mean = float(np.sqrt(np.mean(mean_diff ** 2)))
    axes[3].text(0.97, 0.04,
                 f'MAE  = {mae_mean:.3f}\nRMSE = {rmse_mean:.3f}',
                 transform=axes[3].transAxes, fontsize=9,
                 ha='right', va='bottom', color='white',
                 bbox=dict(fc='#1a1a2e', ec='#555577',
                           alpha=0.9, pad=4, boxstyle='round'))

    fig.suptitle(
        f'Mean rainfall over {n_samples} validation samples  |  '
        '0.5° input  →  0.1° downscaling',
        fontsize=13, fontweight='bold', color='white', y=1.02
    )
    plt.tight_layout(w_pad=0.5)
    plt.savefig(save_path, dpi=250, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    print(f"✅  Mean strip saved → {save_path}")
    plt.close()


# ── RUN ────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("STEP 11 — 0.5° vs 0.1° RAINFALL VISUALIZATIONS (10 SAMPLES)")
print("="*70)

model.load_state_dict(torch.load('best_unet_rainfall.pt'))

print("\n[1/2] Individual 25-sample comparison grid...")
plot_05_vs_01_comparison(
    model, val_loader, device,
    lat_coords_01  = lat_coords,
    lon_coords_01  = lon_coords,
    era5_india_05  = era5_india_05,
    rain_05_downsampled = rain_05_downsampled,
    n_samples      =25,
    save_path      = 'rainfall_05_vs_01_comparison.png'
)

print("\n[2/2] Time-mean strip (mean over 25 samples)...")
plot_mean_rainfall_strip(
    model, val_loader, device,
    lat_coords_01  = lat_coords,
    lon_coords_01  = lon_coords,
    era5_india_05  = era5_india_05,
    rain_05_downsampled = rain_05_downsampled,
    n_samples      = 25,
    save_path      = 'rainfall_mean_strip.png'
)

print("\n✅  STEP 11 complete!")
print("  rainfall_05_vs_01_comparison.png — 10×4 panel grid")
print("  rainfall_mean_strip.png          — 4-panel mean / bias strip")


STEP 11 — 0.5° vs 0.1° RAINFALL VISUALIZATIONS (10 SAMPLES)

[1/2] Individual 25-sample comparison grid...


/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_ocean.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/usr/local/lib/python3.12/dist-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


✅  0.5° vs 0.1° comparison saved → rainfall_05_vs_01_comparison.png

[2/2] Time-mean strip (mean over 25 samples)...
✅  Mean strip saved → rainfall_mean_strip.png

✅  STEP 11 complete!
  rainfall_05_vs_01_comparison.png — 10×4 panel grid
  rainfall_mean_strip.png          — 4-panel mean / bias strip
